In [1]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

INFILE = Path("POPNOVA.xlsx")
OUTFILE = Path("POPNOVA_interpolada_2007_2023.xlsx")

SHEETS = None

def padronizar_colunas_ano(df):
    df = df.copy()
    rename = {}

    for col in df.columns:
        s = str(col).strip()

        if re.fullmatch(r"\d{4}", s):
            rename[col] = int(s)

        elif re.fullmatch(r"\d{4}\.0", s):
            rename[col] = int(float(s))

    return df.rename(columns=rename)


def converter_populacao_para_numero(serie):
    if pd.api.types.is_numeric_dtype(serie):
        return pd.to_numeric(serie, errors="coerce")

    s = (
        serie.astype(str)
        .str.strip()
        .str.replace(".", "", regex=False)
        .str.replace(",", ".", regex=False)
    )

    s = s.replace({"": np.nan, "nan": np.nan, "None": np.nan})
    return pd.to_numeric(s, errors="coerce")


def interpolar_cagr(pop_a, pop_b, ano_a, ano_b, ano_t):
    pop_a = pd.to_numeric(pop_a, errors="coerce")
    pop_b = pd.to_numeric(pop_b, errors="coerce")

    expoente = (ano_t - ano_a) / (ano_b - ano_a)

    resultado = np.where(
        (pop_a > 0) & (pop_b > 0),
        pop_a * (pop_b / pop_a) ** expoente,
        np.nan
    )

    return pd.Series(resultado, index=pop_a.index)


def ordenar_colunas(df):
    colunas_ano = sorted([c for c in df.columns if isinstance(c, int)])
    colunas_nao_ano = [c for c in df.columns if not isinstance(c, int)]
    return df[colunas_nao_ano + colunas_ano]


xls = pd.ExcelFile(INFILE)

if SHEETS is None:
    SHEETS = xls.sheet_names

abas_processadas = {}
resumo = []

for sheet in SHEETS:
    df = pd.read_excel(INFILE, sheet_name=sheet)
    df = padronizar_colunas_ano(df)

    colunas_ano = [c for c in df.columns if isinstance(c, int)]

    for ano in colunas_ano:
        df[ano] = converter_populacao_para_numero(df[ano])

    if 2006 in df.columns and 2008 in df.columns:
        df[2007] = interpolar_cagr(
            pop_a=df[2006],
            pop_b=df[2008],
            ano_a=2006,
            ano_b=2008,
            ano_t=2007
        )
        df[2007] = df[2007].round().astype("Int64")
        status_2007 = "2007 interpolado por CAGR entre 2006 e 2008"
    else:
        status_2007 = "2007 não interpolado: faltam 2006 e/ou 2008"

    if 2022 in df.columns and 2024 in df.columns and df[2022].notna().sum() > 0:
        df[2023] = interpolar_cagr(
            pop_a=df[2022],
            pop_b=df[2024],
            ano_a=2022,
            ano_b=2024,
            ano_t=2023
        )
        df[2023] = df[2023].round().astype("Int64")
        status_2023 = "2023 interpolado por CAGR entre 2022 e 2024"

    elif 2021 in df.columns and 2024 in df.columns:
        df[2023] = interpolar_cagr(
            pop_a=df[2021],
            pop_b=df[2024],
            ano_a=2021,
            ano_b=2024,
            ano_t=2023
        )
        df[2023] = df[2023].round().astype("Int64")
        status_2023 = "2023 interpolado por CAGR entre 2021 e 2024, pois 2022 está ausente ou vazio"

    else:
        status_2023 = "2023 não interpolado: faltam pontos de referência adequados"

    df = ordenar_colunas(df)

    abas_processadas[sheet] = df

    resumo.append({
        "aba": sheet,
        "linhas": len(df),
        "status_2007": status_2007,
        "valores_2007_preenchidos": int(df[2007].notna().sum()) if 2007 in df.columns else 0,
        "status_2023": status_2023,
        "valores_2023_preenchidos": int(df[2023].notna().sum()) if 2023 in df.columns else 0
    })


with pd.ExcelWriter(OUTFILE, engine="openpyxl") as writer:
    for sheet, df in abas_processadas.items():
        df.to_excel(writer, sheet_name=sheet, index=False)

resumo_df = pd.DataFrame(resumo)

print("Arquivo salvo em:", OUTFILE)
display(resumo_df)

Arquivo salvo em: POPNOVA_interpolada_2007_2023.xlsx


,aba,linhas,status_2007,valores_2007_preenchidos,status_2023,valores_2023_preenchidos
0,POP-SIDRA-RJ,92,2007 interpolado por CAGR entre 2006 e 2008,92,2023 interpolado por CAGR entre 2022 e 2024,92
1,POP-SIDRA-SP,645,2007 interpolado por CAGR entre 2006 e 2008,645,2023 interpolado por CAGR entre 2022 e 2024,643


In [2]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

INFILE = Path("POPNOVA_interpolada.xlsx")
OUTFILE = Path("POPNOVA_interpolada_SP_2022_2023.xlsx")

def padronizar_colunas_ano(df):
    df = df.copy()
    rename = {}

    for col in df.columns:
        s = str(col).strip()

        if re.fullmatch(r"\d{4}", s):
            rename[col] = int(s)

        elif re.fullmatch(r"\d{4}\.0", s):
            rename[col] = int(float(s))

    return df.rename(columns=rename)

def converter_populacao_para_numero(serie):
    if pd.api.types.is_numeric_dtype(serie):
        return pd.to_numeric(serie, errors="coerce")

    s = (
        serie.astype(str)
        .str.strip()
        .str.replace(".", "", regex=False)
        .str.replace(",", ".", regex=False)
    )

    s = s.replace({"": np.nan, "nan": np.nan, "None": np.nan})
    return pd.to_numeric(s, errors="coerce")

def interpolar_cagr(pop_a, pop_b, ano_a, ano_b, ano_t):
    pop_a = pd.to_numeric(pop_a, errors="coerce")
    pop_b = pd.to_numeric(pop_b, errors="coerce")

    expoente = (ano_t - ano_a) / (ano_b - ano_a)

    resultado = np.where(
        (pop_a > 0) & (pop_b > 0),
        pop_a * (pop_b / pop_a) ** expoente,
        np.nan
    )

    return pd.Series(resultado, index=pop_a.index)

def ordenar_colunas(df):
    colunas_ano = sorted([c for c in df.columns if isinstance(c, int)])
    colunas_nao_ano = [c for c in df.columns if not isinstance(c, int)]
    return df[colunas_nao_ano + colunas_ano]

xls = pd.ExcelFile(INFILE)
print("Abas encontradas:", xls.sheet_names)

abas_processadas = {}

for sheet in xls.sheet_names:
    df = pd.read_excel(INFILE, sheet_name=sheet)
    df = padronizar_colunas_ano(df)

    colunas_ano = [c for c in df.columns if isinstance(c, int)]

    for ano in colunas_ano:
        df[ano] = converter_populacao_para_numero(df[ano])

    if "sp" in sheet.lower() or "são paulo" in sheet.lower() or "sao paulo" in sheet.lower():
        print(f"Interpolando 2022 e 2023 na aba: {sheet}")

        if 2021 not in df.columns or 2024 not in df.columns:
            raise ValueError(f"A aba {sheet} precisa ter as colunas 2021 e 2024 para interpolar 2022 e 2023.")

        df[2022] = interpolar_cagr(
            pop_a=df[2021],
            pop_b=df[2024],
            ano_a=2021,
            ano_b=2024,
            ano_t=2022
        )

        df[2023] = interpolar_cagr(
            pop_a=df[2021],
            pop_b=df[2024],
            ano_a=2021,
            ano_b=2024,
            ano_t=2023
        )

        df[2022] = df[2022].round().astype("Int64")
        df[2023] = df[2023].round().astype("Int64")

        print("Valores preenchidos:")
        print("2022:", df[2022].notna().sum())
        print("2023:", df[2023].notna().sum())

    df = ordenar_colunas(df)
    abas_processadas[sheet] = df

with pd.ExcelWriter(OUTFILE, engine="openpyxl") as writer:
    for sheet, df in abas_processadas.items():
        df.to_excel(writer, sheet_name=sheet, index=False)

print("Arquivo salvo em:", OUTFILE)

Abas encontradas: ['POP-SIDRA-RJ', 'POP-SIDRA-SP']
Interpolando 2022 e 2023 na aba: POP-SIDRA-SP
Valores preenchidos:
2022: 645
2023: 645
Arquivo salvo em: POPNOVA_interpolada_SP_2022_2023.xlsx
